# Gold Price Prediction — COT + Macro Features
**Pipeline:** Daily gold price (GC=F) predicted using CFTC COT positioning, macro indicators (DXY, real yields, VIX), and technical features.  
**Method mirrors:** APS1052H/option_6/option6_v2_timed.ipynb (COT → log(CADUSD))  
**Key differences:** daily frequency, gold COT filter, DXY + real-yield features, model saving with joblib/keras.


## Analytical Pipeline
```
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  1. DATA         │────▶│  2. STATIONARITY │────▶│  3. FEATURES     │
│  Gold (GC=F)    │     │  ADF + KPSS      │     │  COT + Macro +TA │
│  COT (GOLD)     │     │  Frac Diff d≈0.3 │     │  ~35 features    │
│  DXY/VIX/TNX   │     └──────────────────┘     └──────────────────┘
└─────────────────┘                                        │
         ┌─────────────────────────────────────────────────┘
         ▼
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  4. SPLIT 80/20 │────▶│  5. CV LOOP 1    │────▶│  6. CV LOOP 2    │
│  Chronological  │     │  9 models        │     │  Hyperparameter  │
│  No lookahead   │     │  TimeSeriesSplit  │     │  GridSearch      │
└─────────────────┘     └──────────────────┘     └─────────┬────────┘
                                                             │
         ┌───────────────────────────────────────────────────┘
         ▼
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  7. TEST EVAL   │────▶│  8. TRADING      │────▶│  9. SAVE MODELS  │
│  MSE + charts   │     │  CAGR/Sharpe/    │     │  joblib + keras  │
│                 │     │  PF / MaxDD      │     │                  │
└─────────────────┘     └──────────────────┘     └──────────────────┘
         │
         ▼
┌─────────────────┐     ┌──────────────────┐
│  10. SHAP       │────▶│  11. WRC + MCPT  │
│  Feature imp.   │     │  Selection bias  │
└─────────────────┘     └──────────────────┘
```


## 0. Install Dependencies (run once)
```bash
pip install yfinance pandas numpy matplotlib seaborn scikit-learn xgboost shap
pip install tensorflow pmdarima cot_reports pandas-ta joblib statsmodels tqdm
```


In [ ]:
# ── Progress Tracking ────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
import time

_cell_times = {}
_TOTAL_CELLS = 28
_pbar = tqdm(total=_TOTAL_CELLS, desc='Pipeline', unit='cell')


In [ ]:
_t0 = time.time()  # ⏱ timing: 1. IMPORTS
# ── 1. IMPORTS ───────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
import datetime

# Data
import yfinance as yf

# Stats
from statsmodels.tsa.stattools import adfuller, kpss

# Sklearn
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

# XGBoost
import xgboost as xgb

# Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

# TA
try:
    import talib
    TALIB_AVAILABLE = True
    print('TA-Lib loaded ✓')
except ImportError:
    TALIB_AVAILABLE = False
    print('TA-Lib not found — using pandas-ta fallback')
    import pandas_ta as ta

# SHAP
import shap

# ARIMA
try:
    import pmdarima as pm
    PMDARIMA_AVAILABLE = True
    print('pmdarima loaded ✓')
except ImportError:
    PMDARIMA_AVAILABLE = False
    print('pmdarima not available — ARIMAX will be skipped')

SEED = 42
np.random.seed(SEED)

RESULTS_DIR = 'results'
MODELS_DIR  = 'saved_models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

print('All imports OK ✓')
_cell_times['1. IMPORTS'] = time.time() - _t0
print(f'⏱ 1. IMPORTS: {_cell_times["1. IMPORTS"]:.2f}s')
_pbar.update(1)


## 2. Data Download
### 2a. Gold (GC=F) + Macro Indicators from Yahoo Finance
| Ticker | Series | Role |
|--------|--------|------|
| GC=F   | Gold futures continuous | Target |
| DX-Y.NYB | USD Index (DXY) | Inverse macro driver |
| ^VIX   | CBOE Volatility Index | Risk-off proxy |
| ^TNX   | US 10-Year Treasury Yield | Rate environment |
| ^TYX   | US 30-Year Treasury Yield | Long-end rates |


In [ ]:
_t0 = time.time()  # ⏱ timing: 2a. GOLD + MACRO DOWNLOAD
# ── 2a. GOLD + MACRO DOWNLOAD ────────────────────────────────────────────────
START_DATE = datetime.datetime(2006, 1, 1)
END_DATE   = datetime.datetime.now()

# Gold futures daily close
gold_raw = yf.download('GC=F', start=START_DATE, end=END_DATE, interval='1d', auto_adjust=True)
gold = gold_raw[['Close']].copy()
gold.columns = ['Gold']
gold.index = pd.to_datetime(gold.index).normalize()
gold.dropna(inplace=True)
gold['log_Gold'] = np.log(gold['Gold'])
print(f'Gold rows: {len(gold)} | {gold.index[0].date()} → {gold.index[-1].date()}')

# Download macro series
def _download(ticker, col_name, label):
    try:
        raw = yf.download(ticker, start=START_DATE, end=END_DATE, interval='1d', auto_adjust=True)
        if raw.empty:
            print(f'  WARNING: {label} download empty')
            return None
        s = raw[['Close']].copy()
        s.columns = [col_name]
        s.index = pd.to_datetime(s.index).normalize()
        print(f'  {label}: {len(s)} rows | {s.index[0].date()} → {s.index[-1].date()}')
        return s
    except Exception as e:
        print(f'  {label} failed: {e}')
        return None

print('Downloading macro indicators...')
dxy  = _download('DX-Y.NYB', 'DXY',        'DXY (USD Index)')
vix  = _download('^VIX',     'VIX',        'VIX')
tnx  = _download('^TNX',     'TNX_yield',  'TNX (10Y yield)')
tyx  = _download('^TYX',     'TYX_yield',  'TYX (30Y yield)')

gold.to_csv('gold_daily.csv')
print('Gold data saved ✓')

_cell_times['2a. GOLD + MACRO DOWNLOAD'] = time.time() - _t0
print(f'⏱ 2a: {_cell_times["2a. GOLD + MACRO DOWNLOAD"]:.2f}s')
_pbar.update(1)


### 2b. COT Data — CFTC Gold Futures
Filter: `GOLD` in Market_and_Exchange_Names (COMEX gold futures).  
COT is reported weekly (Tuesday). We forward-fill to daily.


In [ ]:
_t0 = time.time()  # ⏱ timing: 2b. COT GOLD
# ── 2b. COT DATA — GOLD FUTURES ──────────────────────────────────────────────
COT_COL_MAP = {
    'Market and Exchange Names':            'Market_and_Exchange_Names',
    'Commercial Positions-Long (All)':      'Comm_Positions_Long_All',
    'Commercial Positions-Short (All)':     'Comm_Positions_Short_All',
    'Noncommercial Positions-Long (All)':   'NonComm_Positions_Long_All',
    'Noncommercial Positions-Short (All)':  'NonComm_Positions_Short_All',
    'Open Interest (All)':                  'Open_Interest_All',
    'Nonreportable Positions-Long (All)':   'NonRept_Positions_Long_All',
    'Nonreportable Positions-Short (All)':  'NonRept_Positions_Short_All',
    'As of Date in Form YYMMDD':            'As_of_Date_in_Form_YYMMDD',
    'As of Date in Form YYYY-MM-DD':        'As_of_Date_in_Form_YYYY_MM_DD',
}

def _normalize_cot_cols(df):
    return df.rename(columns={k: v for k, v in COT_COL_MAP.items() if k in df.columns})

cot_gold = None

try:
    import cot_reports as cot
    year_end = datetime.datetime.now().year
    frames = []
    for y in range(2006, year_end + 1):
        try:
            df_y = cot.cot_year(year=y, cot_report_type='legacy_fut', store_txt=False, verbose=False)
            frames.append(_normalize_cot_cols(df_y))
        except Exception as e:
            print(f'  skip {y}: {e}')
    if frames:
        cot_raw = pd.concat(frames, ignore_index=True)
        print(f'cot_reports: loaded {len(frames)} years ✓')
        
        # Filter to GOLD futures (COMEX)
        gold_mask = cot_raw['Market_and_Exchange_Names'].astype(str).str.contains(
            r'GOLD', case=False, na=False
        )
        # Exclude 'MINI GOLD' if present — keep full-size contract only
        mini_mask = cot_raw['Market_and_Exchange_Names'].astype(str).str.contains(
            r'MINI', case=False, na=False
        )
        cot_gold_raw = cot_raw[gold_mask & ~mini_mask].copy()
        print(f'Unique gold contract names found:')
        print(cot_gold_raw['Market_and_Exchange_Names'].unique())
        
        # Parse date
        if 'As_of_Date_in_Form_YYYY_MM_DD' in cot_gold_raw.columns:
            cot_gold_raw['Date'] = pd.to_datetime(cot_gold_raw['As_of_Date_in_Form_YYYY_MM_DD'], errors='coerce')
        else:
            cot_gold_raw['Date'] = pd.to_datetime(
                cot_gold_raw['As_of_Date_in_Form_YYMMDD'].astype(str).str.replace('.0','',regex=False),
                format='%y%m%d', errors='coerce')

        cot_gold_raw = cot_gold_raw.dropna(subset=['Date']).drop_duplicates(subset=['Date'], keep='last')
        cot_gold_raw = cot_gold_raw.set_index('Date').sort_index()

        keep_cols = [
            'Comm_Positions_Long_All', 'Comm_Positions_Short_All',
            'NonComm_Positions_Long_All', 'NonComm_Positions_Short_All',
            'Open_Interest_All',
            'NonRept_Positions_Long_All', 'NonRept_Positions_Short_All',
        ]
        cot_gold = cot_gold_raw[[c for c in keep_cols if c in cot_gold_raw.columns]]
        print(f'COT Gold rows: {len(cot_gold)} | {cot_gold.index[0].date()} → {cot_gold.index[-1].date()}')
    else:
        print('No COT data loaded')
except Exception as e:
    print(f'cot_reports not available: {e}')
    print('Will proceed without COT data (reduced feature set)')

_cell_times['2b. COT GOLD'] = time.time() - _t0
print(f'⏱ 2b: {_cell_times["2b. COT GOLD"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 2c. MERGE ALL SOURCES
# ── 2c. MERGE GOLD + COT + MACRO (daily) ─────────────────────────────────────
# COT is weekly (Tuesday release). Strategy: resample to daily, forward-fill.
# This preserves causal ordering — no lookahead.

df = gold.copy()
df.index = pd.to_datetime(df.index).normalize()

# Merge COT (weekly → daily ffill)
if cot_gold is not None and len(cot_gold) > 0:
    # Reindex to daily, then ffill — each trading day gets the most recent COT report
    cot_daily = cot_gold.reindex(df.index, method='ffill')
    df = df.join(cot_daily, how='left')
    print(f'COT columns merged (ffilled): {list(cot_gold.columns)}')
    print(f'COT null count after merge: {df[cot_gold.columns].isnull().sum().sum()}')
    HAS_COT = True
else:
    HAS_COT = False
    print('No COT data — will use macro + TA features only')

# Merge macro series (daily, left join + ffill)
for series, label in [(dxy, 'DXY'), (vix, 'VIX'), (tnx, 'TNX_yield'), (tyx, 'TYX_yield')]:
    if series is not None:
        series.index = pd.to_datetime(series.index).normalize()
        df = df.join(series, how='left')
        col = series.columns[0]
        df[col] = df[col].ffill()
        print(f'{label} merged: {df[col].notna().sum()} non-null rows')

# Drop rows where macro data is missing (early history)
df.dropna(subset=['log_Gold'], inplace=True)
df.to_csv('merged_gold_dataset.csv')
print(f'\nMerged shape: {df.shape}')
print(f'Date range:   {df.index[0].date()} → {df.index[-1].date()}')
print(f'Columns:      {list(df.columns)}')
df.tail(3)

_cell_times['2c. MERGE ALL SOURCES'] = time.time() - _t0
print(f'⏱ 2c: {_cell_times["2c. MERGE ALL SOURCES"]:.2f}s')
_pbar.update(1)


## 3. Stationarity Check (ADF + KPSS)
Same dual-test approach as the USD/CAD model.  
- **ADF**: H₀ = unit root (non-stationary). p < 0.05 → reject H₀ → stationary.  
- **KPSS**: H₀ = stationary. p > 0.05 → fail to reject → stationary.  
Both must agree for confidence.


In [ ]:
_t0 = time.time()  # ⏱ timing: 3. ADF + KPSS TESTS
# ── 3. ADF + KPSS STATIONARITY TEST ─────────────────────────────────────────
def adf_test(series, name='series'):
    result = adfuller(series.dropna())
    pval = result[1]
    stat = 'STATIONARY ✓' if pval < 0.05 else 'NON-STATIONARY ✗'
    print(f'{name:<35} ADF stat={result[0]:>8.4f}  p={pval:.4f}  → {stat}')
    return pval < 0.05

def kpss_test(series, name='series'):
    try:
        stat, pval, lags, crit = kpss(series.dropna(), regression='c', nlags='auto')
        result = 'STATIONARY (fail to reject H0)' if pval > 0.05 else 'NON-STATIONARY (reject H0)'
        print(f'{name:<35} KPSS stat={stat:>8.4f}  p={pval:.4f}  → {result}')
    except Exception as e:
        print(f'{name:<35} KPSS error: {e}')

print('=== ADF Test ===')
adf_test(df['Gold'],               'Gold (level)')
adf_test(df['log_Gold'],           'log(Gold)')
adf_test(df['log_Gold'].diff().dropna(), 'Δlog(Gold) [simple diff]')

if HAS_COT:
    for col in ['Comm_Positions_Long_All','NonComm_Positions_Long_All','Open_Interest_All']:
        if col in df.columns:
            adf_test(df[col].dropna(), col)

print()
print('=== KPSS Test (H0: stationary) ===')
kpss_test(df['Gold'],               'Gold (level)')
kpss_test(df['log_Gold'],           'log(Gold)')
kpss_test(df['log_Gold'].diff().dropna(), 'Δlog(Gold) [simple diff]')

_cell_times['3. ADF + KPSS TESTS'] = time.time() - _t0
print(f'⏱ 3: {_cell_times["3. ADF + KPSS TESTS"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 3b. FRACTIONAL DIFFERENCING
# ── 3b. FRACTIONAL DIFFERENCING (Lopez de Prado fixed-window) ────────────────
# For daily data, use window=252 (1 trading year) instead of 52 weeks

def frac_diff(series, d, window=252, threshold=1e-5):
    """Fixed-width window fractional differencing."""
    w = [1.0]
    for k in range(1, window):
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold:
            break
        w.append(w_k)
    w = np.array(w[::-1])  # oldest weight first

    n = len(series)
    width = len(w)
    result = np.full(n, np.nan)

    for i in range(width - 1, n):
        window_vals = series.iloc[i - width + 1: i + 1].values
        if not np.any(np.isnan(window_vals)):
            result[i] = np.dot(w, window_vals)

    return pd.Series(result, index=series.index)

# Try d values until stationary (same search as USD/CAD model)
for d in [0.3, 0.4, 0.5, 0.6, 0.7]:
    fd = frac_diff(df['log_Gold'], d=d, window=252)
    is_stat = adf_test(fd.dropna(), f'FracDiff(log_Gold, d={d})')
    if is_stat:
        df['frac_diff_log_Gold'] = fd
        FRAC_D = d
        break
else:
    print('Warning: using d=0.7 (not confirmed stationary)')
    df['frac_diff_log_Gold'] = frac_diff(df['log_Gold'], d=0.7, window=252)
    FRAC_D = 0.7

TARGET = 'frac_diff_log_Gold'
print(f'\n→ TARGET = "{TARGET}" (d={FRAC_D})')

# Also fractionally difference COT series that are non-stationary
if HAS_COT:
    non_stat_cot = ['Comm_Positions_Long_All', 'Comm_Positions_Short_All',
                    'NonComm_Positions_Long_All', 'NonComm_Positions_Short_All',
                    'Open_Interest_All']
    for col in non_stat_cot:
        if col in df.columns:
            is_stat = adf_test(df[col].dropna(), col)
            if not is_stat:
                df[f'{col}_diff'] = frac_diff(df[col], d=FRAC_D, window=252)
                print(f'  → fractionally differenced: {col}_diff')

_cell_times['3b. FRACTIONAL DIFFERENCING'] = time.time() - _t0
print(f'⏱ 3b: {_cell_times["3b. FRACTIONAL DIFFERENCING"]:.2f}s')
_pbar.update(1)


## 4. Feature Engineering
Feature groups:
1. **COT features** — net positioning, % of OI, changes, MAs, normalized indices (0–1)
2. **Macro features** — DXY, 10Y yield, 30Y yield, VIX, yield spread (30Y−10Y)
3. **Technical features** — RSI, ROC, MOM, MACD, Bollinger Band width
4. **Target lags** — fractionally differenced log-gold at t-1..t-5
5. **Log return** — for trading equity curve


In [ ]:
_t0 = time.time()  # ⏱ timing: 4. FEATURE ENGINEERING
# ── 4. FEATURE ENGINEERING ───────────────────────────────────────────────────

# ── A. COT features ──────────────────────────────────────────────────────────
if HAS_COT:
    df['net_commercial']   = df['Comm_Positions_Long_All']    - df['Comm_Positions_Short_All']
    df['net_speculator']   = df['NonComm_Positions_Long_All'] - df['NonComm_Positions_Short_All']
    df['net_nonreport']    = df['NonRept_Positions_Long_All'] - df['NonRept_Positions_Short_All']

    df['comm_pct_OI']      = df['net_commercial'] / df['Open_Interest_All']
    df['spec_pct_OI']      = df['net_speculator'] / df['Open_Interest_All']

    # 5-day and 20-day (≈ 1 and 4 COT-report) MAs of positioning
    df['net_comm_chg']     = df['net_commercial'].diff(5)
    df['net_spec_chg']     = df['net_speculator'].diff(5)
    df['OI_chg']           = df['Open_Interest_All'].diff(5)

    df['comm_ma20']        = df['net_commercial'].rolling(20).mean()
    df['spec_ma20']        = df['net_speculator'].rolling(20).mean()
    df['comm_spec_spread'] = df['net_commercial'] - df['net_speculator']

    # COT Normalized Index — rolling 252-day (1-year) min-max scale to [0,1]
    # Gold-specific: speculators drive price momentum; commercials (miners) hedge
    NORM_WINDOW = 252
    for col_out, raw_col in [('cot_norm_spec', 'net_speculator'), ('cot_norm_comm', 'net_commercial')]:
        roll_min = df[raw_col].rolling(NORM_WINDOW, min_periods=60).min()
        roll_max = df[raw_col].rolling(NORM_WINDOW, min_periods=60).max()
        df[col_out] = (df[raw_col] - roll_min) / (roll_max - roll_min + 1e-12)

    print(f'COT norm spec range: [{df["cot_norm_spec"].min():.3f}, {df["cot_norm_spec"].max():.3f}]')
    print(f'COT norm comm range: [{df["cot_norm_comm"].min():.3f}, {df["cot_norm_comm"].max():.3f}]')

# ── B. Macro features ────────────────────────────────────────────────────────
# DXY: inverse relationship with gold (key predictor)
if 'DXY' in df.columns:
    df['DXY_chg']        = df['DXY'].pct_change(5)      # 5-day % change
    df['DXY_ma20']       = df['DXY'].rolling(20).mean()
    df['DXY_vs_ma20']    = df['DXY'] / df['DXY_ma20'] - 1  # deviation from MA

# Real-yield proxy: TNX - VIX-implied inflation (simple: TNX - 0.5*VIX)
# Better proxy for 10Y real yield (TIPS not on Yahoo daily)
if 'TNX_yield' in df.columns and 'VIX' in df.columns:
    df['yield_spread']   = df.get('TYX_yield', df['TNX_yield']) - df['TNX_yield']  # 30Y-10Y curve
    df['real_yield_px']  = df['TNX_yield'] - df['VIX'] * 0.15  # VIX-adjusted yield proxy

# VIX regime: rolling z-score
if 'VIX' in df.columns:
    vix_roll = df['VIX'].rolling(60)
    df['VIX_zscore'] = (df['VIX'] - vix_roll.mean()) / vix_roll.std()

# ── C. Technical features (price-based) ─────────────────────────────────────
close = df['Gold'].values.astype(float)

if TALIB_AVAILABLE:
    df['RSI_14']        = talib.RSI(close, timeperiod=14)
    df['RSI_28']        = talib.RSI(close, timeperiod=28)
    df['ROC_10']        = talib.ROC(close, timeperiod=10)
    df['ROC_20']        = talib.ROC(close, timeperiod=20)
    df['MOM_5']         = talib.MOM(close, timeperiod=5)
    df['EMA_12']        = talib.EMA(close, timeperiod=12)
    df['EMA_26']        = talib.EMA(close, timeperiod=26)
    df['MACD_diff']     = df['EMA_12'] - df['EMA_26']
    upper, middle, lower = talib.BBANDS(close, timeperiod=20)
    df['BB_width']      = (upper - lower) / (middle + 1e-12)
    df['BB_pct']        = (close - lower) / (upper - lower + 1e-12)
else:
    df['RSI_14']        = ta.rsi(df['Gold'], length=14)
    df['RSI_28']        = ta.rsi(df['Gold'], length=28)
    df['ROC_10']        = df['Gold'].pct_change(10) * 100
    df['ROC_20']        = df['Gold'].pct_change(20) * 100
    df['MOM_5']         = df['Gold'].diff(5)
    df['EMA_12']        = df['Gold'].ewm(span=12).mean()
    df['EMA_26']        = df['Gold'].ewm(span=26).mean()
    df['MACD_diff']     = df['EMA_12'] - df['EMA_26']
    df['BB_width']      = df['Gold'].rolling(20).std() / df['Gold'].rolling(20).mean()
    bb_mid              = df['Gold'].rolling(20).mean()
    bb_std              = df['Gold'].rolling(20).std()
    df['BB_pct']        = (df['Gold'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-12)

# ── D. Target lags (shifted to avoid leakage) ───────────────────────────────
for lag in [1, 2, 3, 5, 10]:
    df[f'target_lag{lag}'] = df[TARGET].shift(lag)

# ── E. Log return (for trading equity curve) ─────────────────────────────────
df['log_return'] = df['log_Gold'].diff()

df.dropna(inplace=True)
print(f'Feature matrix shape after dropna: {df.shape}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')

_cell_times['4. FEATURE ENGINEERING'] = time.time() - _t0
print(f'⏱ 4: {_cell_times["4. FEATURE ENGINEERING"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: FEATURE PREVIEW + COT INDEX PLOT
# ── Feature preview ───────────────────────────────────────────────────────────
cot_features = [
    'net_commercial', 'net_speculator', 'net_nonreport',
    'comm_pct_OI', 'spec_pct_OI',
    'net_comm_chg', 'net_spec_chg', 'OI_chg',
    'comm_ma20', 'spec_ma20', 'comm_spec_spread',
    'cot_norm_spec', 'cot_norm_comm',
] if HAS_COT else []

print(f'COT features:   {len([f for f in cot_features if f in df.columns])}')
print(f'Macro features: ~6 (DXY, VIX, TNX, TYX, yield_spread, real_yield_px, DXY_chg...)')
print(f'TA features:    ~8 (RSI, ROC, MOM, MACD, BB_width, BB_pct...)')
print(f'Target lags:    5')
print(f'Total columns:  {df.shape[1]}')

# Plot COT normalized index
if HAS_COT and 'cot_norm_spec' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(df.index, df['Gold'], color='gold', lw=1.2)
    axes[0].set_title('Gold Price (USD/oz)')
    axes[0].set_ylabel('Price')

    axes[1].plot(df.index, df['cot_norm_spec'], color='steelblue', lw=1, label='Speculator (norm)')
    axes[1].plot(df.index, df['cot_norm_comm'], color='tomato',    lw=1, label='Commercial (norm)')
    axes[1].axhline(0.9, color='gray', ls='--', lw=0.8, alpha=0.7)
    axes[1].axhline(0.1, color='gray', ls='--', lw=0.8, alpha=0.7)
    axes[1].set_title('COT Normalized Index (0=most short, 1=most long in 252-day window)')
    axes[1].set_ylabel('Index')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/cot_normalized_gold.png', dpi=150, bbox_inches='tight')
    plt.show()

_cell_times['FEATURE PREVIEW'] = time.time() - _t0
print(f'⏱ Feature preview: {_cell_times["FEATURE PREVIEW"]:.2f}s')
_pbar.update(1)


## 5. Train / Test Split (80/20 Chronological)
No data leakage: test set is touched **once** at final evaluation.  
Target must NOT appear in feature set (verified below).


In [ ]:
_t0 = time.time()  # ⏱ timing: 5. TRAIN / TEST SPLIT
# ── 5. TRAIN / TEST SPLIT ────────────────────────────────────────────────────

# Build feature columns list
_cot_cols = [
    # Raw COT (stationary ones)
    'NonComm_Positions_Long_All', 'NonRept_Positions_Short_All',
    # Derived COT
    'net_commercial', 'net_speculator', 'net_nonreport',
    'comm_pct_OI', 'spec_pct_OI',
    'net_comm_chg', 'net_spec_chg', 'OI_chg',
    'comm_ma20', 'spec_ma20', 'comm_spec_spread',
    'cot_norm_spec', 'cot_norm_comm',
    # Fractionally differenced COT series
    'Comm_Positions_Long_All_diff', 'Comm_Positions_Short_All_diff',
    'NonComm_Positions_Short_All_diff', 'Open_Interest_All_diff',
] if HAS_COT else []

_macro_cols = [
    'DXY', 'DXY_chg', 'DXY_ma20', 'DXY_vs_ma20',
    'VIX', 'VIX_zscore',
    'TNX_yield', 'TYX_yield',
    'yield_spread', 'real_yield_px',
]

_ta_cols = [
    'RSI_14', 'RSI_28', 'ROC_10', 'ROC_20', 'MOM_5',
    'MACD_diff', 'BB_width', 'BB_pct',
]

_lag_cols = [f'target_lag{lag}' for lag in [1, 2, 3, 5, 10]]

FEATURE_COLS = [c for c in (_cot_cols + _macro_cols + _ta_cols + _lag_cols) if c in df.columns]

# Safety: TARGET must not be in features
assert TARGET not in FEATURE_COLS, f'LEAKAGE: {TARGET} is in FEATURE_COLS!'
print(f'No leakage ✓ — TARGET ({TARGET}) not in features')

X = df[FEATURE_COLS].values
y = df[TARGET].values
dates = df.index

n          = len(X)
train_end  = int(n * 0.80)

X_trainval, y_trainval = X[:train_end], y[:train_end]
X_test,     y_test     = X[train_end:], y[train_end:]
dates_test             = dates[train_end:]
log_ret_test           = df['log_return'].values[train_end:]

print(f'Train+Val: {train_end} rows  |  Test: {len(X_test)} rows')
print(f'Train:     {dates[0].date()} → {dates[train_end-1].date()}')
print(f'Test:      {dates_test[0].date()} → {dates_test[-1].date()}')
print(f'Features:  {len(FEATURE_COLS)}')
for i, f in enumerate(FEATURE_COLS):
    print(f'  {i+1:2d}. {f}')

# Visualize split
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(dates[:train_end], y[:train_end], label='Train+Val', color='steelblue', lw=1)
ax.plot(dates[train_end:], y[train_end:], label='Test',      color='tomato',    lw=1)
ax.set_title(f'Fractionally Differenced log(Gold) — Train/Test Split (d={FRAC_D})')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['5. TRAIN / TEST SPLIT'] = time.time() - _t0
print(f'⏱ 5: {_cell_times["5. TRAIN / TEST SPLIT"]:.2f}s')
_pbar.update(1)


## 6. CV Loop 1 — Model Selection
Compare **9 models** with `TimeSeriesSplit` (5 folds). Metric = MSE on validation fold.

| Model | Type | Expected |
|-------|------|----------|
| Ridge | Linear | Likely best (OU process) |
| Lasso | Linear sparse | |
| SVR (linear) | Linear kernel | |
| SVR (RBF) | Non-linear | |
| XGBoost | Tree | |
| RandomForest | Tree | |
| MLP | Neural net | |
| LSTM | RNN | |
| SimpleRNN | Linear RNN | |
| ARIMAX | Statistical | |


In [ ]:
_t0 = time.time()  # ⏱ timing: 6. CV LOOP 1 — MODEL SELECTION
# ── 6. CV LOOP 1 — MODEL SELECTION ──────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5)
results = {}

def cv_sklearn(name, model, X, y, tscv):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    fold_mses = []
    for tr, va in tscv.split(X):
        pipe.fit(X[tr], y[tr])
        mse = mean_squared_error(y[va], pipe.predict(X[va]))
        fold_mses.append(mse)
    print(f'{name:<22} CV MSE: {np.mean(fold_mses):.8f} ± {np.std(fold_mses):.8f}')
    return fold_mses

print('=== CV Loop 1: Model Comparison ===')
results['Ridge']   = cv_sklearn('Ridge (α=1)',    Ridge(alpha=1.0),                              X_trainval, y_trainval, tscv)
results['Lasso']   = cv_sklearn('Lasso (α=1)',    Lasso(alpha=1.0),                              X_trainval, y_trainval, tscv)
results['SVR_lin'] = cv_sklearn('SVR (linear)',   SVR(kernel='linear', C=1),                    X_trainval, y_trainval, tscv)
results['SVR_rbf'] = cv_sklearn('SVR (RBF)',      SVR(kernel='rbf', C=10),                      X_trainval, y_trainval, tscv)
results['XGB']     = cv_sklearn('XGBoost',        xgb.XGBRegressor(n_estimators=100, max_depth=3,
                                                   learning_rate=0.05, random_state=SEED, verbosity=0),
                                                   X_trainval, y_trainval, tscv)
results['RF']      = cv_sklearn('RandomForest',   RandomForestRegressor(n_estimators=100, random_state=SEED),
                                                   X_trainval, y_trainval, tscv)
results['MLP']     = cv_sklearn('MLP',            MLPRegressor(hidden_layer_sizes=(64, 32),
                                                   max_iter=500, random_state=SEED),
                                                   X_trainval, y_trainval, tscv)

_cell_times['6. CV LOOP 1 — MODEL SELECTION'] = time.time() - _t0
print(f'⏱ 6: {_cell_times["6. CV LOOP 1 — MODEL SELECTION"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 6b. LSTM CV
# ── 6b. LSTM CV — improved: recurrent_dropout + L2 regularization ────────────
TIMESTEPS_CANDIDATES = [3, 5, 10, 20]   # candidate look-back windows

def make_sequences(X, y, timesteps):
    Xs, ys = [], []
    for i in range(timesteps, len(X)):
        Xs.append(X[i - timesteps:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

def build_lstm(units=32, dropout=0.35, recurrent_dropout=0.2,
               n_features=len(FEATURE_COLS), timesteps=5):
    """Improved LSTM: recurrent_dropout + L2 kernel regularizer to reduce overfitting."""
    model = Sequential([
        LSTM(units, input_shape=(timesteps, n_features), return_sequences=False,
             dropout=dropout,
             recurrent_dropout=recurrent_dropout,
             kernel_regularizer=l2(0.001)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Grid search over timestep candidates
lstm_ts_results = {}
scaler_lstm = StandardScaler()

for ts in TIMESTEPS_CANDIDATES:
    fold_mses = []
    for tr, va in tscv.split(X_trainval):
        X_tr_s = scaler_lstm.fit_transform(X_trainval[tr])
        X_va_s = scaler_lstm.transform(X_trainval[va])
        Xtr_seq, ytr_seq = make_sequences(X_tr_s, y_trainval[tr], ts)
        Xva_seq, yva_seq = make_sequences(X_va_s, y_trainval[va], ts)
        if len(Xva_seq) == 0:
            continue
        lstm = build_lstm(units=32, n_features=X_trainval.shape[1], timesteps=ts)
        lstm.fit(Xtr_seq, ytr_seq, epochs=50, batch_size=32, verbose=0,
                 callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
        fold_mses.append(mean_squared_error(yva_seq, lstm.predict(Xva_seq, verbose=0).flatten()))
    lstm_ts_results[ts] = fold_mses
    print(f'  LSTM ts={ts:>2d}  CV MSE: {np.mean(fold_mses):.8f} ± {np.std(fold_mses):.8f}')

LSTM_BEST_TS = min(lstm_ts_results, key=lambda ts: np.mean(lstm_ts_results[ts]))
results['LSTM'] = lstm_ts_results[LSTM_BEST_TS]
print(f'{"LSTM":<22} best_ts={LSTM_BEST_TS}  CV MSE: {np.mean(results["LSTM"]):.8f}')

_cell_times['6b. LSTM CV'] = time.time() - _t0
print(f'⏱ 6b: {_cell_times["6b. LSTM CV"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 6c. GRU CV
# ── 6c. GRU CV — replaces SimpleRNN (GRU: gated unit, no vanishing gradient) ─
# GRU has reset+update gates like LSTM but ~30% fewer params → better for small samples.

def build_gru(units=32, dropout=0.3, recurrent_dropout=0.2,
              n_features=len(FEATURE_COLS), timesteps=5):
    """GRU: gated recurrent unit — outperforms SimpleRNN on financial data."""
    model = Sequential([
        GRU(units, input_shape=(timesteps, n_features),
            dropout=dropout,
            recurrent_dropout=recurrent_dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Grid search over timestep candidates
gru_ts_results = {}
scaler_gru = StandardScaler()

for ts in TIMESTEPS_CANDIDATES:
    fold_mses = []
    for tr, va in tscv.split(X_trainval):
        X_tr_s = scaler_gru.fit_transform(X_trainval[tr])
        X_va_s = scaler_gru.transform(X_trainval[va])
        Xtr_seq, ytr_seq = make_sequences(X_tr_s, y_trainval[tr], ts)
        Xva_seq, yva_seq = make_sequences(X_va_s, y_trainval[va], ts)
        if len(Xva_seq) == 0:
            continue
        gru = build_gru(units=32, n_features=X_trainval.shape[1], timesteps=ts)
        gru.fit(Xtr_seq, ytr_seq, epochs=50, batch_size=32, verbose=0,
                callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
        fold_mses.append(mean_squared_error(yva_seq, gru.predict(Xva_seq, verbose=0).flatten()))
    gru_ts_results[ts] = fold_mses
    print(f'  GRU ts={ts:>2d}  CV MSE: {np.mean(fold_mses):.8f} ± {np.std(fold_mses):.8f}')

GRU_BEST_TS = min(gru_ts_results, key=lambda ts: np.mean(gru_ts_results[ts]))
results['GRU'] = gru_ts_results[GRU_BEST_TS]
print(f'{"GRU":<22} best_ts={GRU_BEST_TS}  CV MSE: {np.mean(results["GRU"]):.8f}')

_cell_times['6c. GRU CV'] = time.time() - _t0
print(f'⏱ 6c: {_cell_times["6c. GRU CV"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 6d. BiLSTM CV
# ── 6d. BiLSTM CV — gold-specific: macro regime shifts benefit from BiLSTM ───
# Bidirectional LSTM reads sequence forward AND backward.
# Literature: BiLSTM validated for gold price prediction (DXY, yield regime patterns).

def build_bilstm(units=32, dropout=0.4, recurrent_dropout=0.2,
                 n_features=len(FEATURE_COLS), timesteps=5):
    """BiLSTM: bidirectional LSTM — literature-backed for gold/macro time series."""
    model = Sequential([
        Bidirectional(LSTM(units, recurrent_dropout=recurrent_dropout),
                      input_shape=(timesteps, n_features)),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Grid search over timestep candidates
bilstm_ts_results = {}
scaler_bilstm = StandardScaler()

for ts in TIMESTEPS_CANDIDATES:
    fold_mses = []
    for tr, va in tscv.split(X_trainval):
        X_tr_s = scaler_bilstm.fit_transform(X_trainval[tr])
        X_va_s = scaler_bilstm.transform(X_trainval[va])
        Xtr_seq, ytr_seq = make_sequences(X_tr_s, y_trainval[tr], ts)
        Xva_seq, yva_seq = make_sequences(X_va_s, y_trainval[va], ts)
        if len(Xva_seq) == 0:
            continue
        bilstm = build_bilstm(units=32, n_features=X_trainval.shape[1], timesteps=ts)
        bilstm.fit(Xtr_seq, ytr_seq, epochs=50, batch_size=32, verbose=0,
                   callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
        fold_mses.append(mean_squared_error(yva_seq, bilstm.predict(Xva_seq, verbose=0).flatten()))
    bilstm_ts_results[ts] = fold_mses
    print(f'  BiLSTM ts={ts:>2d}  CV MSE: {np.mean(fold_mses):.8f} ± {np.std(fold_mses):.8f}')

BILSTM_BEST_TS = min(bilstm_ts_results, key=lambda ts: np.mean(bilstm_ts_results[ts]))
results['BiLSTM'] = bilstm_ts_results[BILSTM_BEST_TS]
print(f'{"BiLSTM":<22} best_ts={BILSTM_BEST_TS}  CV MSE: {np.mean(results["BiLSTM"]):.8f}')

_cell_times['6d. BiLSTM CV'] = time.time() - _t0
print(f'⏱ 6d: {_cell_times["6d. BiLSTM CV"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 6e. ARIMAX CV
# ── 6d. ARIMAX CV ────────────────────────────────────────────────────────────
if PMDARIMA_AVAILABLE:
    arimax_fold_mses = []
    scaler_arimax = StandardScaler()
    for tr, va in tscv.split(X_trainval):
        try:
            X_tr_s = scaler_arimax.fit_transform(X_trainval[tr])
            X_va_s = scaler_arimax.transform(X_trainval[va])
            arima_m = pm.auto_arima(y_trainval[tr], exogenous=X_tr_s,
                                    seasonal=False, stepwise=True, suppress_warnings=True,
                                    max_p=3, max_q=3, max_d=1, error_action='ignore', trace=False)
            preds = arima_m.predict(n_periods=len(va), exogenous=X_va_s)
            arimax_fold_mses.append(mean_squared_error(y_trainval[va], preds))
        except Exception as e:
            print(f'  ARIMAX fold failed: {e}')
    if arimax_fold_mses:
        results['ARIMAX'] = arimax_fold_mses
        print(f'{"ARIMAX":<22} CV MSE: {np.mean(arimax_fold_mses):.8f} ± {np.std(arimax_fold_mses):.8f}')
else:
    print('ARIMAX skipped (pmdarima not installed)')

_cell_times['6e. ARIMAX CV'] = time.time() - _t0
print(f'⏱ 6e: {_cell_times["6d. ARIMAX CV"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 6e. PICK BEST MODEL
# ── 6e. PICK BEST MODEL ──────────────────────────────────────────────────────
mean_mses = {name: np.mean(mses) for name, mses in results.items()}
BEST_MODEL_NAME = min(mean_mses, key=mean_mses.get)

print('\n=== Model Selection Summary ===')
for name, mse in sorted(mean_mses.items(), key=lambda x: x[1]):
    marker = '  <-- BEST' if name == BEST_MODEL_NAME else ''
    print(f'  {name:<22} {mse:.8f}{marker}')

# Bar charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
names = list(mean_mses.keys())
mses_vals = [mean_mses[n] for n in names]
colors = ['tomato' if n == BEST_MODEL_NAME else 'steelblue' for n in names]

axes[0].bar(names, mses_vals, color=colors)
axes[0].set_yscale('log')
axes[0].set_title('CV MSE by Model (log scale)')
axes[0].set_ylabel('Mean CV MSE (log)')
axes[0].tick_params(axis='x', rotation=45)

non_keras = {n: m for n, m in mean_mses.items() if n not in ['LSTM', 'GRU', 'BiLSTM', 'ARIMAX']}
axes[1].bar(non_keras.keys(), non_keras.values(),
            color=['tomato' if n == BEST_MODEL_NAME else 'steelblue' for n in non_keras])
axes[1].set_title('CV MSE — sklearn + ARIMAX (linear scale)')
axes[1].set_ylabel('Mean CV MSE')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/model_comparison_cv_mse.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['6e. PICK BEST MODEL'] = time.time() - _t0
print(f'⏱ 6e: {_cell_times["6e. PICK BEST MODEL"]:.2f}s')
_pbar.update(1)
# ── Resolve TIMESTEPS for downstream use ─────────────────────────────────────
# Each RNN model carries its own best timesteps from CV Loop 1.
# Non-RNN models set TIMESTEPS to the best LSTM value (used in all-model comparison).
if BEST_MODEL_NAME == 'LSTM':
    TIMESTEPS = LSTM_BEST_TS
elif BEST_MODEL_NAME == 'GRU':
    TIMESTEPS = GRU_BEST_TS
elif BEST_MODEL_NAME == 'BiLSTM':
    TIMESTEPS = BILSTM_BEST_TS
else:
    TIMESTEPS = LSTM_BEST_TS  # default for all-model Keras comparison
print(f'TIMESTEPS resolved to {TIMESTEPS} (LSTM_BEST_TS={LSTM_BEST_TS}, GRU_BEST_TS={GRU_BEST_TS}, BILSTM_BEST_TS={BILSTM_BEST_TS})')


## 7. CV Loop 2 — Feature Selection + Hyperparameter Tuning
Uses the best model from Loop 1.  
- Inner loop: `SelectKBest` with k ∈ {10, 20, 30, all}  
- Outer loop: model-specific grid search  
- Validation metric: MSE on 5-fold `TimeSeriesSplit`


In [ ]:
_t0 = time.time()  # ⏱ timing: 7. CV LOOP 2 — FEATURE SEL + HYPERPARAM TUNING
# ── 7. CV LOOP 2 ─────────────────────────────────────────────────────────────
from itertools import product

if BEST_MODEL_NAME in ['XGB', 'XGBoost']:
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [2, 3, 4], 'learning_rate': [0.01, 0.05, 0.1]}
elif BEST_MODEL_NAME == 'Ridge':
    param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
elif BEST_MODEL_NAME == 'Lasso':
    param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]}
elif BEST_MODEL_NAME in ['SVR_rbf', 'SVR_lin']:
    param_grid = {'C': [0.1, 1, 10, 100], 'epsilon': [0.001, 0.01, 0.1]}
elif BEST_MODEL_NAME == 'RF':
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, None]}
elif BEST_MODEL_NAME == 'MLP':
    param_grid = {'hidden_layer_sizes': [(32,), (64, 32), (128, 64)], 'alpha': [0.001, 0.01]}
elif BEST_MODEL_NAME in ['LSTM', 'GRU', 'BiLSTM']:
    # TIMESTEPS is now a hyperparameter alongside units and dropout
    param_grid = {
        'timesteps': TIMESTEPS_CANDIDATES,
        'units':     [16, 32, 64],
        'dropout':   [0.2, 0.3, 0.4],
    }
else:
    param_grid = {}

k_values = [10, 20, 30, len(FEATURE_COLS)]
tscv2 = TimeSeriesSplit(n_splits=5)

best_score        = np.inf
best_params       = {}
best_k            = len(FEATURE_COLS)
best_feature_mask = None

print(f'Running CV Loop 2 for {BEST_MODEL_NAME}...')

for k in k_values:
    selector = SelectKBest(score_func=mutual_info_regression, k=min(k, X_trainval.shape[1]))
    X_sel    = selector.fit_transform(X_trainval, y_trainval)

    if not param_grid:
        break

    keys = list(param_grid.keys())
    for vals in product(*param_grid.values()):
        params = dict(zip(keys, vals))
        fold_mses = []

        for tr, va in tscv2.split(X_sel):
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(X_sel[tr])
            Xva = scaler.transform(X_sel[va])

            if BEST_MODEL_NAME in ['XGB', 'XGBoost']:
                m = xgb.XGBRegressor(
                    n_estimators=params['n_estimators'],
                    max_depth=params['max_depth'],
                    learning_rate=params['learning_rate'],
                    random_state=SEED, verbosity=0)
            elif BEST_MODEL_NAME == 'Ridge':
                m = Ridge(alpha=params['alpha'])
            elif BEST_MODEL_NAME == 'Lasso':
                m = Lasso(alpha=params['alpha'])
            elif BEST_MODEL_NAME == 'SVR_rbf':
                m = SVR(kernel='rbf', C=params['C'], epsilon=params['epsilon'])
            elif BEST_MODEL_NAME == 'SVR_lin':
                m = SVR(kernel='linear', C=params['C'], epsilon=params['epsilon'])
            elif BEST_MODEL_NAME == 'MLP':
                m = MLPRegressor(hidden_layer_sizes=params['hidden_layer_sizes'],
                                 alpha=params['alpha'], max_iter=500, random_state=SEED)
            elif BEST_MODEL_NAME == 'RF':
                m = RandomForestRegressor(n_estimators=params['n_estimators'],
                                          max_depth=params['max_depth'], random_state=SEED)
            elif BEST_MODEL_NAME == 'LSTM':
                ts = params['timesteps']
                Xtr_seq, ytr_seq = make_sequences(Xtr, y_trainval[tr], ts)
                Xva_seq, yva_seq = make_sequences(Xva, y_trainval[va], ts)
                if len(Xva_seq) == 0:
                    continue
                m = build_lstm(units=params['units'], dropout=params['dropout'],
                               n_features=X_sel.shape[1], timesteps=ts)
                m.fit(Xtr_seq, ytr_seq, epochs=20, batch_size=32, verbose=0,
                      callbacks=[EarlyStopping(patience=3, restore_best_weights=True)])
                fold_mses.append(mean_squared_error(yva_seq, m.predict(Xva_seq, verbose=0).flatten()))
                continue  # skip the generic m.fit below
            elif BEST_MODEL_NAME == 'GRU':
                ts = params['timesteps']
                Xtr_seq, ytr_seq = make_sequences(Xtr, y_trainval[tr], ts)
                Xva_seq, yva_seq = make_sequences(Xva, y_trainval[va], ts)
                if len(Xva_seq) == 0:
                    continue
                m = build_gru(units=params['units'], dropout=params['dropout'],
                              n_features=X_sel.shape[1], timesteps=ts)
                m.fit(Xtr_seq, ytr_seq, epochs=30, batch_size=32, verbose=0,
                      callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
                fold_mses.append(mean_squared_error(yva_seq, m.predict(Xva_seq, verbose=0).flatten()))
                continue  # skip the generic m.fit below
            elif BEST_MODEL_NAME == 'BiLSTM':
                ts = params['timesteps']
                Xtr_seq, ytr_seq = make_sequences(Xtr, y_trainval[tr], ts)
                Xva_seq, yva_seq = make_sequences(Xva, y_trainval[va], ts)
                if len(Xva_seq) == 0:
                    continue
                m = build_bilstm(units=params['units'], dropout=params['dropout'],
                                 n_features=X_sel.shape[1], timesteps=ts)
                m.fit(Xtr_seq, ytr_seq, epochs=30, batch_size=32, verbose=0,
                      callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
                fold_mses.append(mean_squared_error(yva_seq, m.predict(Xva_seq, verbose=0).flatten()))
                continue  # skip the generic m.fit below
            else:
                m = Ridge()

            m.fit(Xtr, y_trainval[tr])
            fold_mses.append(mean_squared_error(y_trainval[va], m.predict(Xva)))

        if not fold_mses:
            continue
        score = np.mean(fold_mses)
        if score < best_score:
            best_score        = score
            best_params       = params
            best_k            = k
            best_feature_mask = selector.get_support()

# Update TIMESTEPS if best model is RNN-based
if BEST_MODEL_NAME in ['LSTM', 'GRU', 'BiLSTM'] and 'timesteps' in best_params:
    TIMESTEPS = best_params['timesteps']
    print(f'CV Loop 2 updated TIMESTEPS → {TIMESTEPS}')

print(f'\nBest k={best_k} | Best params={best_params} | CV MSE={best_score:.8f}')
if best_feature_mask is not None:
    selected_features = [FEATURE_COLS[i] for i, s in enumerate(best_feature_mask) if s]
    print(f'Selected features: {selected_features}')
else:
    selected_features = FEATURE_COLS

_cell_times['7. CV LOOP 2'] = time.time() - _t0
print(f'⏱ 7: {_cell_times["7. CV LOOP 2"]:.2f}s')
_pbar.update(1)


## 8. Final Test Evaluation (MSE)


In [ ]:
_t0 = time.time()  # ⏱ timing: 8. FINAL MODEL — FIT ON TRAINVAL, EVAL ON TEST
# ── 8. FINAL MODEL ───────────────────────────────────────────────────────────
# TIMESTEPS is already resolved (CV Loop 1 + CV Loop 2 may have updated it).

# Apply feature selection
if best_feature_mask is not None:
    X_tv_final = X_trainval[:, best_feature_mask]
    X_te_final = X_test[:, best_feature_mask]
    feat_names_final = selected_features
else:
    X_tv_final = X_trainval
    X_te_final = X_test
    feat_names_final = FEATURE_COLS

# Scale
final_scaler = StandardScaler()
X_tv_s = final_scaler.fit_transform(X_tv_final)
X_te_s = final_scaler.transform(X_te_final)

# Build final model
if BEST_MODEL_NAME in ['XGB', 'XGBoost']:
    final_model = xgb.XGBRegressor(
        n_estimators=best_params.get('n_estimators', 100),
        max_depth=best_params.get('max_depth', 3),
        learning_rate=best_params.get('learning_rate', 0.05),
        random_state=SEED, verbosity=0)
elif BEST_MODEL_NAME == 'Ridge':
    final_model = Ridge(alpha=best_params.get('alpha', 1.0))
elif BEST_MODEL_NAME == 'Lasso':
    final_model = Lasso(alpha=best_params.get('alpha', 0.01))
elif BEST_MODEL_NAME == 'SVR_rbf':
    final_model = SVR(kernel='rbf', C=best_params.get('C', 10),
                      epsilon=best_params.get('epsilon', 0.01))
elif BEST_MODEL_NAME == 'SVR_lin':
    final_model = SVR(kernel='linear', C=best_params.get('C', 1),
                      epsilon=best_params.get('epsilon', 0.01))
elif BEST_MODEL_NAME == 'MLP':
    final_model = MLPRegressor(
        hidden_layer_sizes=best_params.get('hidden_layer_sizes', (64, 32)),
        alpha=best_params.get('alpha', 0.01), max_iter=500, random_state=SEED)
elif BEST_MODEL_NAME == 'RF':
    final_model = RandomForestRegressor(
        n_estimators=best_params.get('n_estimators', 100),
        max_depth=best_params.get('max_depth', None), random_state=SEED)
elif BEST_MODEL_NAME == 'ARIMAX':
    final_model = pm.auto_arima(y_trainval, exogenous=X_tv_s, seasonal=False,
                                stepwise=True, suppress_warnings=True,
                                max_p=3, max_q=3, max_d=1, error_action='ignore', trace=False)
    y_pred_test = final_model.predict(n_periods=len(y_test), exogenous=X_te_s)
    test_mse    = mean_squared_error(y_test, y_pred_test)
elif BEST_MODEL_NAME == 'LSTM':
    # TIMESTEPS already updated by CV Loop 2
    print(f'Building LSTM with timesteps={TIMESTEPS}, units={best_params.get("units", 32)}')
    final_model = build_lstm(units=best_params.get('units', 32),
                             dropout=best_params.get('dropout', 0.2),
                             n_features=X_tv_final.shape[1], timesteps=TIMESTEPS)
    Xtv_seq, ytv_seq = make_sequences(X_tv_s, y_trainval, TIMESTEPS)
    Xte_seq, yte_seq = make_sequences(X_te_s, y_test,     TIMESTEPS)
    final_model.fit(Xtv_seq, ytv_seq, epochs=50, batch_size=32, verbose=0,
                    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    y_pred_test = final_model.predict(Xte_seq, verbose=0).flatten()
    test_mse    = mean_squared_error(yte_seq, y_pred_test)
elif BEST_MODEL_NAME == 'GRU':
    print(f'Building GRU with timesteps={TIMESTEPS}, units={best_params.get("units", 32)}')
    final_model = build_gru(units=best_params.get('units', 32),
                            dropout=best_params.get('dropout', 0.3),
                            n_features=X_tv_final.shape[1], timesteps=TIMESTEPS)
    Xtv_seq, ytv_seq = make_sequences(X_tv_s, y_trainval, TIMESTEPS)
    Xte_seq, yte_seq = make_sequences(X_te_s, y_test,     TIMESTEPS)
    final_model.fit(Xtv_seq, ytv_seq, epochs=50, batch_size=32, verbose=0,
                    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
    y_pred_test = final_model.predict(Xte_seq, verbose=0).flatten()
    test_mse    = mean_squared_error(yte_seq, y_pred_test)
elif BEST_MODEL_NAME == 'BiLSTM':
    print(f'Building BiLSTM with timesteps={TIMESTEPS}, units={best_params.get("units", 32)}')
    final_model = build_bilstm(units=best_params.get('units', 32),
                               dropout=best_params.get('dropout', 0.4),
                               n_features=X_tv_final.shape[1], timesteps=TIMESTEPS)
    Xtv_seq, ytv_seq = make_sequences(X_tv_s, y_trainval, TIMESTEPS)
    Xte_seq, yte_seq = make_sequences(X_te_s, y_test,     TIMESTEPS)
    final_model.fit(Xtv_seq, ytv_seq, epochs=50, batch_size=32, verbose=0,
                    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
    y_pred_test = final_model.predict(Xte_seq, verbose=0).flatten()
    test_mse    = mean_squared_error(yte_seq, y_pred_test)
else:
    final_model = Ridge()

if BEST_MODEL_NAME not in ['LSTM', 'GRU', 'BiLSTM', 'ARIMAX']:
    final_model.fit(X_tv_s, y_trainval)
    y_pred_test = final_model.predict(X_te_s)
    test_mse    = mean_squared_error(y_test, y_pred_test)

print(f'\n=== FINAL MODEL: {BEST_MODEL_NAME} (TIMESTEPS={TIMESTEPS}) ===')
print(f'Test MSE:  {test_mse:.8f}')
print(f'Test RMSE: {np.sqrt(test_mse):.8f}')

# Predicted vs Actual
n_pred = len(y_pred_test)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates_test[:n_pred], y_test[:n_pred],  label='Actual', lw=1.5)
ax.plot(dates_test[:n_pred], y_pred_test,       label='Predicted', lw=1.5, ls='--', alpha=0.8)
ax.set_title(f'{BEST_MODEL_NAME} (ts={TIMESTEPS}) — Predicted vs Actual {TARGET} (Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['8. FINAL MODEL'] = time.time() - _t0
print(f'⏱ 8: {_cell_times["8. FINAL MODEL"]:.2f}s')
_pbar.update(1)


## 9. Trading Metrics — Equity Curve, CAGR, Sharpe, Profit Factor, MaxDD
Signal = z-score threshold on predictions (same approach as USD/CAD model).  
| Metric | Definition |
|--------|------------|
| **CAGR** | (Final equity)^(252/n_days) − 1 |
| **Sharpe** | Mean(daily ret) / Std(daily ret) × √252 |
| **Profit Factor** | Sum(gains) / Sum(|losses|) |
| **Max Drawdown** | Max peak-to-trough decline |


In [ ]:
_t0 = time.time()  # ⏱ timing: 9. TRADING METRICS
# ── 9. TRADING METRICS ───────────────────────────────────────────────────────
actual   = y_test[:n_pred]
pred     = y_pred_test
log_ret  = log_ret_test[:n_pred]
tdates   = dates_test[:n_pred]

# Rolling z-score signal (reduces noise vs raw sign)
ps           = pd.Series(pred)
pred_zscore  = (ps - ps.rolling(5, min_periods=1).mean()) / ps.rolling(5, min_periods=1).std().fillna(1e-8)
signal       = np.where(pred_zscore > 0, 1, -1).astype(float)

print(f'Signal distribution: Long={(signal==1).sum()}, Short={(signal==-1).sum()}')

strategy_log_ret = signal * log_ret
equity_strategy  = np.exp(np.cumsum(strategy_log_ret))
equity_bnh       = np.exp(np.cumsum(log_ret))

# ── Metric functions ─────────────────────────────────────────────────────────
def profit_factor(log_returns):
    r = pd.Series(log_returns)
    gains  = r[r > 0].sum()
    losses = abs(r[r < 0].sum())
    return float(gains / losses) if losses > 0 else np.inf

def cagr(equity_curve, n_days):
    total_years = n_days / 252
    return float(equity_curve[-1] ** (1 / total_years) - 1)

def sharpe_ratio(log_returns, periods_per_year=252):
    pct_ret = np.exp(log_returns) - 1
    return float(pct_ret.mean() / pct_ret.std() * np.sqrt(periods_per_year))

def max_drawdown(equity_curve):
    peak = np.maximum.accumulate(equity_curve)
    dd   = (equity_curve - peak) / peak
    return float(dd.min())

pf       = profit_factor(strategy_log_ret)
cagr_v   = cagr(equity_strategy, len(strategy_log_ret))
sr       = sharpe_ratio(strategy_log_ret)
mdd, _   = max_drawdown(equity_strategy), None
mdd      = max_drawdown(equity_strategy)

pf_bnh   = profit_factor(log_ret)
cagr_bnh = cagr(equity_bnh, len(log_ret))
sr_bnh   = sharpe_ratio(log_ret)
mdd_bnh  = max_drawdown(equity_bnh)

print(f'\n=== Test Period Performance ===')
print(f'{"Metric":<20} {"Strategy":>12} {"Buy&Hold":>12}')
print('-' * 46)
print(f'{"CAGR":<20} {cagr_v:>12.2%} {cagr_bnh:>12.2%}')
print(f'{"Sharpe Ratio":<20} {sr:>12.4f} {sr_bnh:>12.4f}')
print(f'{"Profit Factor":<20} {pf:>12.4f} {pf_bnh:>12.4f}')
print(f'{"Max Drawdown":<20} {mdd:>12.2%} {mdd_bnh:>12.2%}')

# Equity curve plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(tdates, equity_strategy, label='Strategy', color='steelblue', lw=1.5)
axes[0].plot(tdates, equity_bnh,      label='Buy & Hold Gold', color='gold', lw=1.5, ls='--')
axes[0].set_title(f'Equity Curve — Test Period | CAGR={cagr_v:.1%} | Sharpe={sr:.2f} | PF={pf:.2f}')
axes[0].legend()
axes[0].set_ylabel('Cumulative Return')

axes[1].bar(tdates, strategy_log_ret,
            color=['steelblue' if r >= 0 else 'tomato' for r in strategy_log_ret], width=1)
axes[1].set_title('Daily Strategy Log Returns')
axes[1].axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/equity_curve_test.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['9. TRADING METRICS'] = time.time() - _t0
print(f'⏱ 9: {_cell_times["9. TRADING METRICS"]:.2f}s')
_pbar.update(1)


## 9b. All-Model Trading Metrics Comparison
Re-train every model on full train+val set, compute all trading metrics, compare.


In [ ]:
_t0 = time.time()  # ⏱ timing: 9b. ALL-MODEL TRADING METRICS
# ── 9b. ALL-MODEL TRADING METRICS ────────────────────────────────────────────

def _trading_metrics(preds, log_rets):
    n    = len(preds)
    ps   = pd.Series(preds)
    z    = (ps - ps.rolling(5, min_periods=1).mean()) / ps.rolling(5, min_periods=1).std().fillna(1e-8)
    sig  = np.where(z > 0, 1, -1).astype(float)
    strat = sig * np.array(log_rets[:n])
    eq   = np.exp(np.cumsum(strat))
    cagr_v   = float(eq[-1] ** (252 / n) - 1)
    sharpe_v = float(np.mean(strat) / (np.std(strat) + 1e-12) * np.sqrt(252))
    peak     = np.maximum.accumulate(eq)
    mdd_v    = float(((eq - peak) / peak).min())
    g        = strat[strat > 0].sum()
    l        = abs(strat[strat < 0].sum())
    pf_v     = float(g / l) if l > 0 else np.inf
    mse_v    = mean_squared_error(y_test[:n], preds)
    return dict(MSE=mse_v, CAGR=cagr_v, Sharpe=sharpe_v, PF=pf_v, MaxDD=mdd_v, equity=eq, signal=sig)

_sc  = StandardScaler()
_Xtv = _sc.fit_transform(X_trainval)
_Xte = _sc.transform(X_test)

all_metrics = {}

_sk_models = {
    'Ridge':        Ridge(alpha=best_params.get('alpha', 0.01) if BEST_MODEL_NAME=='Ridge' else 0.01),
    'Lasso':        Lasso(alpha=0.001),
    'SVR_lin':      SVR(kernel='linear', C=1),
    'SVR_rbf':      SVR(kernel='rbf', C=10),
    'XGBoost':      xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05,
                                     random_state=SEED, verbosity=0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=SEED),
    'MLP':          MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=SEED),
}
for name, model in _sk_models.items():
    model.fit(_Xtv, y_trainval)
    all_metrics[name] = _trading_metrics(model.predict(_Xte), log_ret_test)
    print(f'  {name} ✓')

# LSTM — use its own best timesteps from CV Loop 1
_Xtr_sq_l, _ytr_sq_l = make_sequences(_Xtv, y_trainval, LSTM_BEST_TS)
_Xte_sq_l, _yte_sq_l = make_sequences(_Xte, y_test,     LSTM_BEST_TS)
_lstm2 = build_lstm(units=32, n_features=_Xtv.shape[1], timesteps=LSTM_BEST_TS)
_lstm2.fit(_Xtr_sq_l, _ytr_sq_l, epochs=30, batch_size=32, verbose=0,
           callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
all_metrics['LSTM'] = _trading_metrics(
    _lstm2.predict(_Xte_sq_l, verbose=0).flatten(), log_ret_test[LSTM_BEST_TS:])
print(f'  LSTM (ts={LSTM_BEST_TS}) ✓')

# GRU — use its own best timesteps from CV Loop 1
_Xtr_sq_g, _ytr_sq_g = make_sequences(_Xtv, y_trainval, GRU_BEST_TS)
_Xte_sq_g, _yte_sq_g = make_sequences(_Xte, y_test,     GRU_BEST_TS)
_gru2 = build_gru(units=32, n_features=_Xtv.shape[1], timesteps=GRU_BEST_TS)
_gru2.fit(_Xtr_sq_g, _ytr_sq_g, epochs=50, batch_size=32, verbose=0,
          callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
all_metrics['GRU'] = _trading_metrics(
    _gru2.predict(_Xte_sq_g, verbose=0).flatten(), log_ret_test[GRU_BEST_TS:])
print(f'  GRU (ts={GRU_BEST_TS}) ✓')

# BiLSTM — use its own best timesteps from CV Loop 1
_Xtr_sq_b, _ytr_sq_b = make_sequences(_Xtv, y_trainval, BILSTM_BEST_TS)
_Xte_sq_b, _yte_sq_b = make_sequences(_Xte, y_test,     BILSTM_BEST_TS)
_bilstm2 = build_bilstm(units=32, n_features=_Xtv.shape[1], timesteps=BILSTM_BEST_TS)
_bilstm2.fit(_Xtr_sq_b, _ytr_sq_b, epochs=50, batch_size=32, verbose=0,
             callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
all_metrics['BiLSTM'] = _trading_metrics(
    _bilstm2.predict(_Xte_sq_b, verbose=0).flatten(), log_ret_test[BILSTM_BEST_TS:])
print(f'  BiLSTM (ts={BILSTM_BEST_TS}) ✓')

# ARIMAX
if PMDARIMA_AVAILABLE:
    try:
        _arima2 = pm.auto_arima(y_trainval, exogenous=_Xtv, seasonal=False,
                                stepwise=True, suppress_warnings=True,
                                max_p=3, max_q=3, max_d=1, error_action='ignore', trace=False)
        all_metrics['ARIMAX'] = _trading_metrics(
            _arima2.predict(n_periods=len(y_test), exogenous=_Xte), log_ret_test)
        print('  ARIMAX ✓')
    except Exception as e:
        print(f'  ARIMAX failed: {e}')

# ── Summary table ────────────────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {k: v for k, v in m.items() if k not in ('equity', 'signal')}
    for name, m in all_metrics.items()
}).T

metrics_df = metrics_df.sort_values('Sharpe', ascending=False)
print('\n=== All-Model Trading Metrics (sorted by Sharpe) ===')
print(metrics_df.to_string(float_format=lambda x: f'{x:.4f}'))
metrics_df.to_csv(f'{RESULTS_DIR}/all_model_metrics.csv')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = metrics_df.index.tolist()
for ax, metric, color in zip(axes, ['Sharpe', 'CAGR', 'PF'], ['steelblue', 'seagreen', 'darkorange']):
    vals = metrics_df[metric].values.astype(float)
    bar_colors = [color if v > 0 else 'tomato' for v in vals]
    ax.bar(model_names, vals, color=bar_colors)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(0, color='black', lw=0.5)

plt.suptitle(f'All-Model Trading Metrics | LSTM ts={LSTM_BEST_TS}, GRU ts={GRU_BEST_TS}, BiLSTM ts={BILSTM_BEST_TS}', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_model_trading_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['9b. ALL-MODEL TRADING METRICS'] = time.time() - _t0
print(f'⏱ 9b: {_cell_times["9b. ALL-MODEL TRADING METRICS"]:.2f}s')
_pbar.update(1)


## 10. Save Models
- sklearn / XGBoost → `joblib`
- Keras (LSTM, SimpleRNN) → `.keras` native format
- Scaler → joblib (needed for inference)
- Feature list → JSON


In [ ]:
_t0 = time.time()  # ⏱ timing: 10. SAVE MODELS
# ── 10. SAVE MODELS ──────────────────────────────────────────────────────────
import json

# Save the final best model
if BEST_MODEL_NAME in ['LSTM', 'GRU', 'BiLSTM']:
    keras_path = f'{MODELS_DIR}/best_model_{BEST_MODEL_NAME}.keras'
    final_model.save(keras_path)
    print(f'Keras model saved: {keras_path}')
elif BEST_MODEL_NAME == 'ARIMAX':
    joblib.dump(final_model, f'{MODELS_DIR}/best_model_ARIMAX.joblib')
    print(f'ARIMAX model saved: {MODELS_DIR}/best_model_ARIMAX.joblib')
else:
    joblib.dump(final_model, f'{MODELS_DIR}/best_model_{BEST_MODEL_NAME}.joblib')
    print(f'sklearn model saved: {MODELS_DIR}/best_model_{BEST_MODEL_NAME}.joblib')

# Save the final scaler
joblib.dump(final_scaler, f'{MODELS_DIR}/final_scaler.joblib')
print(f'Scaler saved: {MODELS_DIR}/final_scaler.joblib')

# Save all models from all-model comparison
for name, model in _sk_models.items():
    joblib.dump(model, f'{MODELS_DIR}/model_{name}.joblib')

_lstm2.save(f'{MODELS_DIR}/model_LSTM.keras')
_gru2.save(f'{MODELS_DIR}/model_GRU.keras')
_bilstm2.save(f'{MODELS_DIR}/model_BiLSTM.keras')
joblib.dump(_sc, f'{MODELS_DIR}/all_models_scaler.joblib')

# Save metadata (includes per-model best timesteps)
meta = {
    'target':            TARGET,
    'frac_d':            FRAC_D,
    'best_model':        BEST_MODEL_NAME,
    'best_params':       {k: (int(v) if isinstance(v, (np.integer,)) else v)
                          for k, v in best_params.items()},
    'best_k':            best_k,
    'feature_cols':      FEATURE_COLS,
    'selected_features': selected_features,
    'timesteps_candidates': TIMESTEPS_CANDIDATES,
    'lstm_best_timesteps':    LSTM_BEST_TS,
    'gru_best_timesteps':     GRU_BEST_TS,
    'bilstm_best_timesteps':  BILSTM_BEST_TS,
    'timesteps':         TIMESTEPS,          # final best model's timesteps
    'has_cot':           HAS_COT,
    'train_end_date':    str(dates[train_end - 1].date()),
    'test_start_date':   str(dates_test[0].date()),
}
with open(f'{MODELS_DIR}/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nAll models saved to {MODELS_DIR}/')
print(f'Metadata: lstm_best_timesteps={LSTM_BEST_TS}, gru_best_timesteps={GRU_BEST_TS}, bilstm_best_timesteps={BILSTM_BEST_TS}')

import os
saved = os.listdir(MODELS_DIR)
for s in sorted(saved):
    size = os.path.getsize(f'{MODELS_DIR}/{s}') / 1024
    print(f'  {s:<40} {size:>8.1f} KB')

_cell_times['10. SAVE MODELS'] = time.time() - _t0
print(f'⏱ 10: {_cell_times["10. SAVE MODELS"]:.2f}s')
_pbar.update(1)


## 11. SHAP Feature Importance


In [ ]:
_t0 = time.time()  # ⏱ timing: 11. SHAP
# ── 11. SHAP ─────────────────────────────────────────────────────────────────
print('Computing SHAP values...')

if BEST_MODEL_NAME in ['XGBoost', 'RandomForest', 'RF']:
    explainer  = shap.TreeExplainer(final_model)
    shap_vals  = explainer.shap_values(X_te_s)
elif BEST_MODEL_NAME in ['LSTM', 'GRU', 'BiLSTM', 'ARIMAX']:
    # Use Ridge as proxy for SHAP when Keras/ARIMA model is best
    print('Note: using Ridge as SHAP proxy for Keras/ARIMA model')
    _proxy = Ridge(alpha=0.01)
    _proxy.fit(X_tv_s, y_trainval)
    background = shap.kmeans(X_tv_s, 20)
    explainer  = shap.KernelExplainer(_proxy.predict, background)
    shap_vals  = explainer.shap_values(X_te_s[:100])
else:
    background = shap.kmeans(X_tv_s, 20)
    explainer  = shap.KernelExplainer(final_model.predict, background)
    shap_vals  = explainer.shap_values(X_te_s[:100])

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_te_s[:len(shap_vals)],
                  feature_names=feat_names_final, show=False)
plt.title(f'SHAP Feature Importance — {BEST_MODEL_NAME} (Test Set)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plot saved ✓')

_cell_times['11. SHAP'] = time.time() - _t0
print(f'⏱ 11: {_cell_times["11. SHAP"]:.2f}s')
_pbar.update(1)


## 12. Selection Bias Tests
### 12a. White Reality Check (WRC)
Block bootstrap under H₀: strategy mean return ≤ 0.  
### 12b. Monte Carlo Permutation Test (MCPT)
Shuffle signals under H₀: signal is random. Unbiased p-value (Phipson & Smyth 2010).


In [ ]:
_t0 = time.time()  # ⏱ timing: 12a. WHITE REALITY CHECK
# ── 12a. WHITE REALITY CHECK ─────────────────────────────────────────────────
def white_reality_check(strategy_returns, n_bootstrap=1000, block_size=10, seed=42):
    rng           = np.random.default_rng(seed)
    observed_mean = strategy_returns.mean()
    n             = len(strategy_returns)
    null_means    = []
    for _ in range(n_bootstrap):
        blocks = []
        while sum(len(b) for b in blocks) < n:
            start = rng.integers(0, n)
            blocks.append(np.take(strategy_returns, range(start, start + block_size), mode='wrap'))
        null_means.append(np.concatenate(blocks)[:n].mean())
    null_means = np.array(null_means)
    return observed_mean, null_means, np.mean(null_means >= observed_mean)

obs_mean, null_dist, wrc_pval = white_reality_check(strategy_log_ret)

print(f'=== White Reality Check ===')
print(f'Observed mean log return: {obs_mean:.8f}')
print(f'WRC p-value: {wrc_pval:.4f}')
print(f'Result: {"REJECT H0 — significant edge ✓" if wrc_pval < 0.05 else "FAIL TO REJECT H0 ✗"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_dist, bins=50, color='steelblue', alpha=0.7, label='Bootstrap null')
ax.axvline(obs_mean, color='tomato', lw=2, label=f'Observed={obs_mean:.5f}')
ax.set_title(f'White Reality Check — p={wrc_pval:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/white_reality_check.png', dpi=150, bbox_inches='tight')
plt.show()

_cell_times['12a. WHITE REALITY CHECK'] = time.time() - _t0
print(f'⏱ 12a: {_cell_times["12a. WHITE REALITY CHECK"]:.2f}s')
_pbar.update(1)


In [ ]:
_t0 = time.time()  # ⏱ timing: 12b. MONTE CARLO PERMUTATION TEST
# ── 12b. MONTE CARLO PERMUTATION TEST ────────────────────────────────────────
def monte_carlo_permutation(signal, log_returns, n_permutations=1000, seed=42):
    rng          = np.random.default_rng(seed)
    observed     = (signal * log_returns).mean()
    perm_means   = [(rng.permutation(signal) * log_returns).mean() for _ in range(n_permutations)]
    perm_means   = np.array(perm_means)
    p_value      = (np.sum(perm_means >= observed) + 1) / (n_permutations + 1)
    return observed, perm_means, p_value

obs_mc, perm_dist, mc_pval = monte_carlo_permutation(signal, log_ret)

print(f'=== Monte Carlo Permutation Test ===')
print(f'Observed mean strategy return: {obs_mc:.8f}')
print(f'MC p-value: {mc_pval:.4f}')
print(f'Result: {"SIGNIFICANT ✓" if mc_pval < 0.05 else "NOT SIGNIFICANT ✗"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_dist, bins=50, color='steelblue', alpha=0.7, label='Permuted null')
ax.axvline(obs_mc, color='tomato', lw=2, label=f'Observed={obs_mc:.5f}')
ax.set_title(f'Monte Carlo Permutation Test — p={mc_pval:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mc_permutation_test.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Final Summary ─────────────────────────────────────────────────────────────
print(f'\n{"=" * 55}')
print('  FINAL SUMMARY')
print(f'{"=" * 55}')
print(f'  Best model:        {BEST_MODEL_NAME}')
print(f'  Features (k):      {best_k} of {len(FEATURE_COLS)}')
print(f'  Frac diff d:       {FRAC_D}')
print(f'  Test MSE:          {test_mse:.8f}')
print(f'  CAGR:              {cagr_v:.2%}')
print(f'  Sharpe Ratio:      {sr:.4f}')
print(f'  Profit Factor:     {pf:.4f}')
print(f'  Max Drawdown:      {mdd:.2%}')
print(f'  WRC p-value:       {wrc_pval:.4f}  {"✓" if wrc_pval < 0.05 else "✗"}')
print(f'  MC p-value:        {mc_pval:.4f}  {"✓" if mc_pval < 0.05 else "✗"}')

# Save summary
with open(f'{RESULTS_DIR}/final_summary.txt', 'w') as f:
    f.write(f'Best model:        {BEST_MODEL_NAME}\n')
    f.write(f'Features (k):      {best_k} of {len(FEATURE_COLS)}\n')
    f.write(f'Frac diff d:       {FRAC_D}\n')
    f.write(f'Test MSE:          {test_mse:.8f}\n')
    f.write(f'CAGR:              {cagr_v:.2%}\n')
    f.write(f'Sharpe Ratio:      {sr:.4f}\n')
    f.write(f'Profit Factor:     {pf:.4f}\n')
    f.write(f'Max Drawdown:      {mdd:.2%}\n')
    f.write(f'WRC p-value:       {wrc_pval:.4f}\n')
    f.write(f'MC p-value:        {mc_pval:.4f}\n')
print(f'Summary saved to {RESULTS_DIR}/final_summary.txt ✓')

_cell_times['12b. MONTE CARLO PERMUTATION TEST'] = time.time() - _t0
print(f'⏱ 12b: {_cell_times["12b. MONTE CARLO PERMUTATION TEST"]:.2f}s')
_pbar.update(1)


In [ ]:
# ── TIMING SUMMARY ────────────────────────────────────────────────────────────
_pbar.close()

_expected = {
    '1. IMPORTS':                     10,
    '2a. GOLD + MACRO DOWNLOAD':       8,
    '2b. COT GOLD':                   25,
    '2c. MERGE ALL SOURCES':           2,
    '3. ADF + KPSS TESTS':             2,
    '3b. FRACTIONAL DIFFERENCING':     5,
    '4. FEATURE ENGINEERING':          2,
    'FEATURE PREVIEW':                 2,
    '5. TRAIN / TEST SPLIT':           2,
    '6. CV LOOP 1 — MODEL SELECTION': 20,
    '6b. LSTM CV':                   120,
    '6c. GRU CV':                     70,
    '6d. BiLSTM CV':                150,
    '6e. ARIMAX CV':                  30,
    '6e. PICK BEST MODEL':             2,
    '7. CV LOOP 2':                   30,
    '8. FINAL MODEL':                  5,
    '9. TRADING METRICS':              2,
    '9b. ALL-MODEL TRADING METRICS':  60,
    '10. SAVE MODELS':                 5,
    '11. SHAP':                       20,
    '12a. WHITE REALITY CHECK':        3,
    '12b. MONTE CARLO PERMUTATION TEST': 3,
}

print('\n' + '=' * 70)
print('  CELL TIMING SUMMARY')
print('=' * 70)
print(f'  {"Section":<40} {"Expected":>9} {"Actual":>9} {"Status":>7}')
print('-' * 70)
for name, t in sorted(_cell_times.items(), key=lambda x: -x[1]):
    exp = _expected.get(name)
    if exp:
        status = 'OK' if t <= exp * 1.5 else 'SLOW'
        print(f'  {name:<40} {exp:>7.0f}s {t:>7.2f}s {status:>7}')
    else:
        print(f'  {name:<40} {"—":>9} {t:>7.2f}s {"—":>7}')
print('-' * 70)
total = sum(_cell_times.values())
exp_total = sum(_expected.values())
print(f'  {"TOTAL":<40} {exp_total:>7.0f}s {total:>7.2f}s')
